# RFM Customer Segmentation Analysis
### Olist Brazilian E-Commerce Platform

**Objective:** Identify distinct customer segments using RFM (Recency, Frequency, Monetary) analysis to prioritise marketing spend and improve customer retention.

**Dataset:** [Olist E-Commerce Public Dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — 100k orders from a Brazilian marketplace (2016–2018), spanning orders, customers, payments, and products.

**Key Finding:** 97% of customers place only one order. This analysis segments the remaining customer base by recency and spend to surface high-value retention opportunities.

---
**Tools:** Python · pandas · SQLite · matplotlib · seaborn

## 1. Setup & Data Loading

The four core tables are loaded from CSV into an in-memory SQLite database. Using SQL for joins and aggregations keeps the logic readable and mirrors real-world analytics workflows.

In [ ]:
# Mount Google Drive (Google Colab)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

# Load raw data
path = '/content/drive/MyDrive/Colab Notebooks/Olist/data/'

orders      = pd.read_csv(path + 'olist_orders_dataset.csv')
order_items = pd.read_csv(path + 'olist_order_items_dataset.csv')
customers   = pd.read_csv(path + 'olist_customers_dataset.csv')
payments    = pd.read_csv(path + 'olist_order_payments_dataset.csv')

# Load into in-memory SQLite
conn = sqlite3.connect(':memory:')

orders.to_sql('orders',           conn, index=False, if_exists='replace')
order_items.to_sql('order_items', conn, index=False, if_exists='replace')
customers.to_sql('customers',     conn, index=False, if_exists='replace')
payments.to_sql('payments',       conn, index=False, if_exists='replace')

print('Tables loaded successfully!')

## 2. Exploratory Data Analysis

### 2.1 Schema Overview

A quick look at available columns across all four tables.

In [ ]:
print('ORDERS:',      orders.columns.tolist())
print('ORDER_ITEMS:', order_items.columns.tolist())
print('CUSTOMERS:',   customers.columns.tolist())
print('PAYMENTS:',    payments.columns.tolist())

### 2.2 Shape & Null Check

Verifying row counts and checking for missing values before any joins or aggregations.

In [ ]:
# Row counts
print('Orders:      ', orders.shape)
print('Order Items: ', order_items.shape)
print('Customers:   ', customers.shape)
print('Payments:    ', payments.shape)

In [ ]:
# Null value audit
print('Orders nulls:\n',    orders.isnull().sum())
print('\nPayments nulls:\n', payments.isnull().sum())
print('\nCustomers nulls:\n', customers.isnull().sum())

### 2.3 Order Status Breakdown

Understanding how many orders reached a completed state is critical before calculating spend — only **delivered** orders represent confirmed revenue.

In [ ]:
pd.read_sql("""
    SELECT order_status, COUNT(*) AS count
    FROM orders
    GROUP BY order_status
    ORDER BY count DESC
""", conn)

### 2.4 Data Cleaning

Filtering to delivered orders only and joining customer and payment data. Rows with undefined payment types or zero-value payments are excluded to avoid distorting monetary calculations.

In [ ]:
clean_orders = pd.read_sql("""
    SELECT
        orders.order_id,
        customers.customer_unique_id,
        orders.order_purchase_timestamp,
        SUM(payments.payment_value) AS total_payment
    FROM orders
    JOIN customers ON orders.customer_id = customers.customer_id
    JOIN payments  ON orders.order_id    = payments.order_id
    WHERE orders.order_status         = 'delivered'
      AND payments.payment_type      != 'not_defined'
      AND payments.payment_value      > 0
    GROUP BY
        orders.order_id,
        customers.customer_unique_id,
        orders.order_purchase_timestamp
""", conn)

print(f'Clean orders: {clean_orders.shape[0]:,} rows, {clean_orders.shape[1]} columns')
clean_orders.head()

## 3. RFM Metric Calculation

RFM is a behavioural segmentation framework that scores each customer on three dimensions:

| Metric | Definition |
|---|---|
| **Recency** | Days since the customer's most recent purchase |
| **Frequency** | Total number of orders placed |
| **Monetary** | Total spend across all orders |

The reference date is **2018-10-01** — one day after the last transaction in the dataset — so recency is calculated relative to the most recent possible point in time.

In [ ]:
# Register clean_orders in SQLite
clean_orders.to_sql('clean_orders', conn, index=False, if_exists='replace')

rfm = pd.read_sql("""
    SELECT
        customer_unique_id,
        CAST(julianday('2018-10-01') - julianday(MAX(order_purchase_timestamp)) AS INTEGER) AS recency,
        COUNT(order_id)       AS frequency,
        SUM(total_payment)    AS monetary
    FROM clean_orders
    GROUP BY customer_unique_id
""", conn)

print(f'Unique customers: {rfm.shape[0]:,}')
rfm.head()

### 3.1 Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

rfm['recency'].hist(bins=30, ax=axes[0], color='steelblue')
axes[0].set_title('Recency Distribution')
axes[0].set_xlabel('Days Since Last Purchase')
axes[0].set_ylabel('Number of Customers')

rfm['frequency'].hist(bins=30, ax=axes[1], color='steelblue')
axes[1].set_title('Frequency Distribution')
axes[1].set_xlabel('Number of Orders')
axes[1].set_ylabel('Number of Customers')

rfm['monetary'].hist(bins=30, ax=axes[2], color='steelblue')
axes[2].set_title('Monetary Distribution')
axes[2].set_xlabel('Total Spend ($)')
axes[2].set_ylabel('Number of Customers')

plt.suptitle('RFM Metric Distributions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
print(rfm[['recency', 'frequency', 'monetary']].describe().round(2))

In [ ]:
print('Frequency breakdown (top 10 values):')
print(rfm['frequency'].value_counts().head(10))

**Key observations:**

- **97% of customers ordered exactly once** (90,556 out of 93,357). Mean frequency is 1.03, making frequency a near-constant across the dataset.
- **Monetary is heavily right-skewed** — median spend is ~107, but the distribution has a long tail up to 13,664.
- **Recency spans 32–727 days**, providing meaningful variation to separate recent from lapsed customers.

*Methodological note: Because frequency has almost no variance, it would add noise rather than signal to the scoring model. The segmentation below uses only Recency and Monetary (RM scoring).*

## 4. RFM Segmentation

Each customer is scored on recency and monetary using **decile ranking** (1–10), then the two scores are summed into a composite `rm_total` (range: 2–20). Higher scores indicate more recent, higher-spending customers.

Segment thresholds were set to produce business-meaningful, roughly proportional groups:

| Segment | RM Score | Description |
|---|---|---|
| Champions | ≥ 17 | Recent, high-spend — your most valuable customers |
| Loyal Customers | 13–16 | Strong base with repeat potential |
| Needs Attention | 9–12 | Largest group — middle-of-road, activatable |
| At Risk | 5–8 | Declining engagement, moderate spend history |
| Lost | < 5 | Lapsed, low-value — minimal investment warranted |

In [ ]:
# Decile scoring: recency (lower days = better) and monetary (higher spend = better)
rfm['r_score'] = pd.qcut(rfm['recency'],  q=10, labels=[10, 9, 8, 7, 6, 5, 4, 3, 2, 1])
rfm['m_score'] = pd.qcut(rfm['monetary'], q=10, labels=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

rfm['rm_total'] = rfm['r_score'].astype(int) + rfm['m_score'].astype(int)

print('RM Total Score — summary stats:')
print(rfm['rm_total'].describe().round(2))

In [ ]:
def assign_segment(score):
    if score >= 17:   return 'Champions'
    elif score >= 13: return 'Loyal Customers'
    elif score >= 9:  return 'Needs Attention'
    elif score >= 5:  return 'At Risk'
    else:             return 'Lost'

rfm['segment'] = rfm['rm_total'].apply(assign_segment)

print('Customer count per segment:')
print(rfm['segment'].value_counts())

## 5. Visualisation

### 5.1 Segment Size & Median Spend

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

segment_order = ['Champions', 'Loyal Customers', 'Needs Attention', 'At Risk', 'Lost']
segment_colors = ['#2ecc71', '#3498db', '#f39c12', '#e67e22', '#e74c3c']

# ── Customer count ─────────────────────────────────────────────────────────────
segment_counts = rfm['segment'].value_counts().reindex(segment_order)
axes[0].bar(segment_counts.index, segment_counts.values, color=segment_colors)
axes[0].set_title('Customer Count by Segment', fontweight='bold')
axes[0].set_xlabel('Segment')
axes[0].set_ylabel('Number of Customers')
axes[0].tick_params(axis='x', rotation=15)

for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 200,
                 f'{int(bar.get_height()):,}',
                 ha='center', fontsize=9)

# ── Median spend ──────────────────────────────────────────────────────────────
segment_monetary = rfm.groupby('segment')['monetary'].median().reindex(segment_order)
axes[1].bar(segment_monetary.index, segment_monetary.values, color=segment_colors)
axes[1].set_title('Median Spend by Segment', fontweight='bold')
axes[1].set_xlabel('Segment')
axes[1].set_ylabel('Median Spend ($)')
axes[1].tick_params(axis='x', rotation=15)

for bar in axes[1].patches:
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 3,
                 f'${bar.get_height():.0f}',
                 ha='center', fontsize=9)

plt.suptitle('Segment Profiles: Size vs. Spend', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Observations:**

- **Champions** (9,631 customers) have a median spend of ~265 — over 6× that of **Lost** customers (~40).
- **Needs Attention** is the largest segment (32,571 customers), representing an untapped activation opportunity.
- **At Risk + Lost** together account for ~28% of the customer base but contribute disproportionately little revenue.

### 5.2 Recency vs. Monetary Scatter by Segment

This plot shows how segments cluster spatially. Champions sit top-left (recent, high-spend); Lost customers sit bottom-right (lapsed, low-spend).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

colors = {
    'Champions':       '#2ecc71',
    'Loyal Customers': '#3498db',
    'Needs Attention': '#f39c12',
    'At Risk':         '#e67e22',
    'Lost':            '#e74c3c'
}

for segment, group in rfm.groupby('segment'):
    ax.scatter(
        group['recency'],
        group['monetary'],
        c=colors[segment],
        label=segment,
        alpha=0.5,
        s=10
    )

ax.set_title('Recency vs. Monetary Value by Segment', fontsize=14, fontweight='bold')
ax.set_xlabel('Recency (Days Since Last Purchase)')
ax.set_ylabel('Monetary Value ($) — Log Scale')
ax.set_yscale('log')
ax.legend(title='Segment', markerscale=2)

plt.tight_layout()
plt.show()

## 6. Business Recommendations

**Context:** Analysis of 93,357 customers across the Olist Brazilian e-commerce platform reveals a critical retention problem — 97% of customers never return after their first order. The recommendations below are prioritised by segment value and are designed to maximise CLV without over-investing in low-return cohorts.

---

### Champions — 9,631 customers | Median spend $265
These are your highest-value, most recent buyers.
- Launch a **VIP loyalty programme** (early access, exclusive offers) to deepen engagement
- Leverage for **product reviews and referrals** — they are the most likely to respond positively
- **Do not discount** — they already buy at full price; discounting erodes margin with no upside

### Loyal Customers — 24,580 customers | Median spend $220
A strong base with clear potential to graduate to Champions.
- **Personalised category recommendations** based on purchase history
- Small incentives (free shipping, 10% off) to trigger a repeat purchase
- Priority segment for **upsell and cross-sell** campaigns

### Needs Attention — 32,571 customers | Median spend $141
The largest segment — middle-of-road behaviour with high activation upside.
- **Time-limited re-engagement email campaign** with a modest offer
- Sub-segment by recency: customers within 6 months are worth more aggressive spend
- A/B test messaging to identify the highest-converting trigger

### At Risk — 20,972 customers | Median spend $71
Engagement is slipping — act before they transition to Lost.
- **Win-back campaign** with a meaningful incentive (15–20% discount)
- Highlight new products or categories not previously purchased
- Set a **hard spend cap** per customer; avoid investing heavily in the low-monetary tail

### Lost — 5,603 customers | Median spend $42
Lapsed, low-value — minimal marketing investment is justified.
- One final win-back email, then **suppress from all future campaigns**
- Reallocate rescued budget to Champions and Loyal Customers

---

### Overall Priority
> **Fix retention before acquisition.** With 97% single-purchase customers, even converting 5% of the Needs Attention segment to Loyal would generate substantial incremental revenue — at zero new customer acquisition cost.